In [41]:
import pandas as pd
import numpy as np

In [42]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

--2026-10-11 02:14:59--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv.1’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.005s  

2026-10-11 02:14:59 (49.7 MB/s) - ‘course_lead_scoring_2026.csv.1’ saved [278746/278746]



In [43]:
df = pd.read_csv("./03-classification/homework/course_lead_scoring_2026.csv")

In [44]:
df.head(20)

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
5,referral,manufacturing,employed,africa,43841.0,3,5,0.77,1
6,NaN,education,employed,north_america,58738.0,3,9,0.78,1
7,paid_ads,technology,employed,north_america,95059.0,2,4,0.41,1
8,organic_search,healthcare,employed,asia,62565.0,3,6,0.52,1
9,referral,retail,student,asia,12000.0,2,2,0.35,0


In [45]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [46]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [47]:
df[['lead_source', 'industry', 'employment_status', 'location']] = df[['lead_source', 'industry', 'employment_status', 'location']].fillna('NA')

In [48]:
df[['annual_income', 'lead_score']] = df[['annual_income', 'lead_score']].fillna(0.0)

In [49]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

### Question 1

In [50]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [51]:
df.industry.value_counts().idxmax()

'technology'

### Question 2

In [52]:
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']
numerical

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score',
 'converted']

In [53]:
correlation_matrix = df[numerical].corr()
correlation_matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


### Question 3

In [54]:
from sklearn.model_selection import train_test_split

In [55]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [56]:
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [57]:
df_train.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
4702,paid_ads,finance,employed,asia,65980.0,2,3,0.37,1
2155,events,finance,employed,south_america,71662.0,0,0,0.09,0
289,referral,NA,employed,asia,40220.0,2,5,0.55,1
1822,organic_search,finance,employed,north_america,88352.0,2,2,0.45,0
3442,events,technology,employed,north_america,89316.0,1,5,0.00,1


In [58]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [59]:
df_train.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,paid_ads,finance,employed,asia,65980.0,2,3,0.37,1
1,events,finance,employed,south_america,71662.0,0,0,0.09,0
2,referral,NA,employed,asia,40220.0,2,5,0.55,1
3,organic_search,finance,employed,north_america,88352.0,2,2,0.45,0
4,events,technology,employed,north_america,89316.0,1,5,0.00,1


In [60]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [61]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [62]:
df_full_train.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
4227,events,healthcare,employed,south_america,47301.0,5,7,0.69,0
4676,organic_search,technology,unemployed,asia,0.0,1,5,0.57,0
800,events,finance,unemployed,asia,47970.0,0,0,0.17,0
3671,organic_search,retail,self_employed,south_america,52847.0,3,3,0.45,0
4193,paid_ads,finance,employed,africa,73460.0,1,4,0.42,1


In [63]:
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
categorical = [col for col in df_full_train.columns if col not in numerical and col != 'converted']
numerical, categorical

(['annual_income',
  'number_of_courses_viewed',
  'interaction_count',
  'lead_score'],
 ['lead_source', 'industry', 'employment_status', 'location'])

In [64]:
from sklearn.metrics import mutual_info_score

In [65]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train.converted)

In [66]:
mi = round(df_full_train[categorical].apply(mutual_info_converted_score), 2)
mi.sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

### Question 4

In [67]:
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

X_train[0 : 2]

array([[6.5980e+04, 0.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 3.0000e+00, 3.7000e-01,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 2.0000e+00],
       [7.1662e+04, 0.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 9.0000e-02,
        0.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 1.0000e+00, 0.0000e+00]])

In [68]:
X_train.shape

(3000, 28)

In [69]:
dv.get_feature_names_out()

array(['annual_income', 'employment_status=NA',
       'employment_status=employed', 'employment_status=self_employed',
       'employment_status=student', 'employment_status=unemployed',
       'industry=NA', 'industry=education', 'industry=finance',
       'industry=healthcare', 'industry=manufacturing', 'industry=retail',
       'industry=technology', 'interaction_count', 'lead_score',
       'lead_source=NA', 'lead_source=events',
       'lead_source=organic_search', 'lead_source=paid_ads',
       'lead_source=referral', 'lead_source=social_media', 'location=NA',
       'location=africa', 'location=asia', 'location=europe',
       'location=north_america', 'location=south_america',
       'number_of_courses_viewed'], dtype=object)

In [70]:
val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [71]:
X_val.shape

(1000, 28)

In [72]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
# solver='lbfgs' is the default solver in newer version of sklearn
# for older versions, you need to specify it explicitly
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [73]:
model.intercept_[0]

np.float64(-0.010838978497809288)

In [74]:
model.coef_[0]

array([-4.57849985e-06, -5.83434850e-04,  7.40346890e-03, -3.28471892e-03,
       -8.04069170e-03, -6.33360193e-03, -4.41226767e-04, -2.95759734e-03,
       -6.49404976e-04, -2.15453817e-03, -2.30751043e-03, -1.65985277e-03,
       -6.68848037e-04,  1.36728058e-01,  1.10036031e-02,  9.51725098e-05,
       -8.62466509e-03,  6.94367905e-03, -1.37825294e-02,  8.71726672e-03,
       -4.18790228e-03,  2.30545170e-04, -1.79050012e-03, -1.77472633e-03,
       -5.83764763e-04, -3.10519687e-03, -3.81533558e-03,  5.37983416e-02])

In [76]:
y_pred = model.predict_proba(X_val)[:, 1]
y_pred.shape

(1000,)

In [77]:
y_pred[:5]

array([0.61725691, 0.44364108, 0.60433996, 0.64562246, 0.52214159])

In [80]:
converted_decision = (y_pred >= 0.5)
converted_decision.shape

(1000,)

In [81]:
y_val.shape

(1000,)

In [83]:
converted_decision.astype(int).shape

(1000,)

In [87]:
(y_val == converted_decision).mean()

np.float64(0.645)

In [88]:
df_pred = pd.DataFrame()
df_pred['probability'] = y_pred
df_pred['prediction'] = converted_decision.astype(int)
df_pred['actual'] = y_val
df_pred

,probability,prediction,actual
0,0.617257,1,1
1,0.443641,0,1
2,0.604340,1,0
3,0.645622,1,1
4,0.522142,1,0
...,...,...,...
995,0.684710,1,1
996,0.669580,1,1
997,0.458473,0,0
998,0.561770,1,0


### Question 5

In [89]:
features = categorical + numerical
features

['lead_source',
 'industry',
 'employment_status',
 'location',
 'annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [96]:
def feature_elimination(feature):
    subset = [f for f in features if f != feature]

    dv = DictVectorizer(sparse=False)

    train_dict = df_train[subset].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    return dv, model, subset

In [98]:
def record_accuracy():

    results = []
    for feature in features:
        dv_sub, model_sub, subset = feature_elimination(feature)

        val_dict = df_val[subset].to_dict(orient='records')
        X_val_sub = dv_sub.transform(val_dict)

        y_pred = model_sub.predict_proba(X_val_sub)[:, 1]
        accuracy = ((y_pred >= 0.5) == y_val).mean()

        results.append((feature, accuracy, 0.645 - accuracy))

    return pd.DataFrame(results, columns=['feature', 'accuracy', 'difference']).sort_values('difference')

In [99]:
record_accuracy()

,feature,accuracy,difference
4,annual_income,0.724,-0.079
1,industry,0.645,0.000
3,location,0.645,0.000
2,employment_status,0.645,0.000
7,lead_score,0.644,0.001
5,number_of_courses_viewed,0.643,0.002
0,lead_source,0.642,0.003
6,interaction_count,0.601,0.044


### Question 6

In [100]:
c = [0.000001, 0.00001, 0.0001, 0.001]

In [102]:
c = [0.000001, 0.00001, 0.0001, 0.001]

param_results = []

dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

for param in c:

    model = LogisticRegression(solver='liblinear', C=param, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    accuracy = ((y_pred >= 0.5) == y_val).mean().round(3)

    param_results.append((param, accuracy))

param_results

[(1e-06, np.float64(0.598)),
 (1e-05, np.float64(0.598)),
 (0.0001, np.float64(0.613)),
 (0.001, np.float64(0.645))]